In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/competitions/finclub-open-project-26/dataset.csv'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
df = pd.read_csv('/kaggle/input/competitions/finclub-open-project-26/dataset.csv')

# Display the top rows to confirm it loaded correctly
display(df.head())

,datetime,underlying_price,NIFTY27JAN2625200CE,NIFTY27JAN2625300CE,NIFTY27JAN2625400CE,NIFTY27JAN2625500CE,NIFTY27JAN2625600CE,NIFTY27JAN2625700CE,NIFTY27JAN2625800CE,NIFTY27JAN2625900CE,...,NIFTY27JAN2624200PE,NIFTY27JAN2624300PE,NIFTY27JAN2624400PE,NIFTY27JAN2624500PE,NIFTY27JAN2624600PE,NIFTY27JAN2624700PE,NIFTY27JAN2624800PE,NIFTY27JAN2624900PE,NIFTY27JAN2625000PE,NIFTY27JAN2625100PE
0,07-01-2026 09:15,26111.65,0.12662,0.12330,0.11741,NaN,0.11005,0.10576,NaN,0.09724,...,0.15760,0.15240,0.14697,0.14105,0.13613,0.13085,0.12640,0.12142,0.11631,0.11150
1,07-01-2026 09:20,26141.40,0.08632,NaN,NaN,0.11779,0.11197,0.11028,NaN,NaN,...,NaN,0.15420,0.14753,0.14274,0.13849,0.13282,NaN,0.12363,NaN,0.11353
2,07-01-2026 09:25,26139.35,0.09147,NaN,0.09514,0.09933,0.09599,0.09204,0.09216,0.08954,...,0.15927,NaN,0.14919,0.14245,0.13806,0.13242,0.12877,0.12349,0.11817,NaN
3,07-01-2026 09:30,26128.95,0.10860,0.10842,0.11150,0.12248,0.10715,0.11098,0.10345,NaN,...,0.15755,NaN,0.14691,0.14209,0.13721,0.13184,0.12722,0.12252,0.11729,0.11200
4,07-01-2026 09:35,26131.90,0.10462,0.10538,0.12459,0.12051,0.11225,0.11294,0.10544,NaN,...,0.15924,0.15334,0.14784,0.14230,NaN,0.13219,0.12733,0.12295,0.11707,NaN


In [4]:
# 1. Structural Transformation: Wide to Long
# Options chains are provided in a wide matrix (Strikes as columns), which prevents cross-sectional feature engineering.
# Melting the matrix anchors the time and underlying price, allowing us to compute spatial distance across strikes.
df_long = pd.melt(
    df, 
    id_vars=['datetime', 'underlying_price'], 
    var_name='ticker', 
    value_name='IV'
)

# 2. Extract features from the ticker string
# Example Ticker: NIFTY27JAN2625200CE
df_long['Type'] = df_long['ticker'].str[-2:]                 # Grabs 'CE' or 'PE'
df_long['Strike'] = df_long['ticker'].str[-7:-2].astype(int) # Grabs '25200' and turns it into a number
df_long['Expiry_Str'] = df_long['ticker'].str[5:-7]          # Grabs '27JAN26'

# 3. Convert strings to actual Time objects
df_long['datetime'] = pd.to_datetime(df_long['datetime'],format='%d-%m-%Y %H:%M')
df_long['Expiry'] = pd.to_datetime(df_long['Expiry_Str'],format='%d%b%y')

# 4. Calculate Time To Expiry (TTE) in annualized years
# (Expiry Date - Current Date) converted to total years
df_long['TTE'] = (df_long['Expiry'] - df_long['datetime']).dt.total_seconds() / (24 * 3600 * 365)

# 5. Sort the data chronologically per ticker so we can apply our baselines
df_long = df_long.drop(columns=['Expiry_Str']) # Drop the string version, we don't need it anymore
df_long = df_long.sort_values(by=['ticker', 'datetime']).reset_index(drop=True)

display(df_long.head(10))


,datetime,underlying_price,ticker,IV,Type,Strike,Expiry,TTE
0,2026-01-07 09:15:00,26111.65,NIFTY27JAN2623800PE,0.17840,PE,23800,2026-01-27,0.053739
1,2026-01-07 09:20:00,26141.40,NIFTY27JAN2623800PE,0.17962,PE,23800,2026-01-27,0.053729
2,2026-01-07 09:25:00,26139.35,NIFTY27JAN2623800PE,0.18010,PE,23800,2026-01-27,0.053720
3,2026-01-07 09:30:00,26128.95,NIFTY27JAN2623800PE,0.18174,PE,23800,2026-01-27,0.053710
4,2026-01-07 09:35:00,26131.90,NIFTY27JAN2623800PE,0.18193,PE,23800,2026-01-27,0.053701
5,2026-01-07 09:40:00,26145.15,NIFTY27JAN2623800PE,0.18277,PE,23800,2026-01-27,0.053691
6,2026-01-07 09:45:00,26113.10,NIFTY27JAN2623800PE,0.18107,PE,23800,2026-01-27,0.053682
7,2026-01-07 09:50:00,26110.00,NIFTY27JAN2623800PE,NaN,PE,23800,2026-01-27,0.053672
8,2026-01-07 09:55:00,26108.90,NIFTY27JAN2623800PE,0.18084,PE,23800,2026-01-27,0.053662
9,2026-01-07 10:00:00,26140.30,NIFTY27JAN2623800PE,0.18280,PE,23800,2026-01-27,0.053653


In [5]:
# Create a copy of our long dataframe to test our baselines safely
df_baseline = df_long.copy()

# 1.  Time-Based Forward Fill
# Group by ticker, sort by time, fill forward.
df_baseline['IV_ffill'] = df_baseline.groupby('ticker')['IV'].fillna(method='ffill')

# 2. Spatial Linear Interpolation
# We sort by time, type, and strike so the strikes line up perfectly.
df_baseline = df_baseline.sort_values(by=['datetime', 'Type', 'Strike'])

# Group by datetime and Type, then interpolate linearly across the Strikes
df_baseline['IV_linear'] = df_baseline.groupby(['datetime', 'Type'])['IV_ffill'].apply(
    lambda group: group.interpolate(method='linear', limit_direction='both')
).reset_index(level=[0,1], drop=True)
# 3. Cubic Spline Interpolation
# We tell Pandas to use a 3rd-order polynomial (spline) to respect the U-shape
df_baseline['IV_spline'] = df_baseline.groupby(['datetime', 'Type'])['IV_ffill'].apply(
    lambda group: group.interpolate(method='cubicspline', limit_direction='both')
).reset_index(level=[0,1], drop=True)
# 3. Check how many missing values are left
missing_original = df_baseline['IV'].isna().sum()
missing_baseline = df_baseline['IV_linear'].isna().sum()

print(f"Original Missing IV values: {missing_original}")
print(f"Missing IV values after baselines: {missing_baseline}")

/tmp/ipykernel_58/302725135.py:6: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df_baseline['IV_ffill'] = df_baseline.groupby('ticker')['IV'].fillna(method='ffill')
/tmp/ipykernel_58/302725135.py:6: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_baseline['IV_ffill'] = df_baseline.groupby('ticker')['IV'].fillna(method='ffill')


Original Missing IV values: 5460
Missing IV values after baselines: 0


In [6]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# 1. Feature Engineering (Basic)
df_ml = df_baseline.copy()

# XGBoost only understands numbers. Convert 'CE' to 1 and 'PE' to 0.
df_ml['Type_Code'] = np.where(df_ml['Type'] == 'CE', 1, 0)

# Define our starting features (X) and our target (y)
features = ['underlying_price', 'Strike', 'TTE', 'Type_Code']
target = 'IV'

# 2. Isolate the "Known" data from the "Missing" data
# We only train and validate on rows where we actually have an IV value
df_known = df_ml[df_ml['IV'].notna()].copy()
df_missing = df_ml[df_ml['IV'].isna()].copy()

# 3. Create Train and Validation Sets (80/20 split)
X = df_known[features]
y = df_known[target]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize and Train XGBoost
# We use standard hyperparameters for our baseline
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")
xgb_model.fit(X_train, y_train)

# 5. Evaluate the Model
y_pred = xgb_model.predict(X_val)
mse = mean_squared_error(y_val, y_pred)

print(f"Baseline XGBoost MSE: {mse:.6f}")

Training XGBoost...
Baseline XGBoost MSE: 0.000580


In [7]:
import numpy as np

# Create a copy to build our advanced feature matrix
df_fe = df_baseline.copy()

#  Add Type_Code back to this specific dataframe
df_fe['Type_Code'] = np.where(df_fe['Type'] == 'CE', 1, 0)

# 1. Financial Metric: Log-Moneyness
df_fe['Moneyness'] = np.log(df_fe['underlying_price'] / df_fe['Strike'])
df_fe['Distance_From_ATM'] = np.abs(df_fe['Moneyness'])

# 2. Expiry Scaling

df_fe['Log_TTE'] = np.log(np.maximum(df_fe['TTE'], 1e-5))

# 3. Structural Cross-Features (Interactions)
df_fe['Strike_TTE_Interaction'] = df_fe['Moneyness'] * df_fe['TTE']

# 4. Spline Guidance Feature
df_fe['Spline_Guess'] = df_fe['IV_spline']

# Let's clean up our feature list
features_advanced = [
    'underlying_price', 'Strike', 'TTE', 'Type_Code',
    'Moneyness', 'Distance_From_ATM', 'Log_TTE', 
    'Strike_TTE_Interaction', 'Spline_Guess'
]

# Display the newly engineered feature matrix
display(df_fe[features_advanced].head())

,underlying_price,Strike,TTE,Type_Code,Moneyness,Distance_From_ATM,Log_TTE,Strike_TTE_Interaction,Spline_Guess
13650,26111.65,25200,0.053739,1,0.035538,0.035538,-2.923624,0.001910,0.126620
14625,26111.65,25300,0.053739,1,0.031577,0.031577,-2.923624,0.001697,0.123300
15600,26111.65,25400,0.053739,1,0.027632,0.027632,-2.923624,0.001485,0.117410
16575,26111.65,25500,0.053739,1,0.023703,0.023703,-2.923624,0.001274,0.113326
17550,26111.65,25600,0.053739,1,0.019789,0.019789,-2.923624,0.001063,0.110050


In [8]:
# Isolate known data with advanced features
df_known_fe = df_fe[df_fe['IV'].notna()].copy()

X_adv = df_known_fe[features_advanced]
y_adv = df_known_fe['IV']

# Use the exact same split parameters for a fair comparison
X_train_a, X_val_a, y_train_a, y_val_a = train_test_split(X_adv, y_adv, test_size=0.2, random_state=42)

# Re-train the model
xgb_advanced = xgb.XGBRegressor(
    n_estimators=150,       # Slightly increased estimators for richer features
    learning_rate=0.08,     # Slower learning rate for smoother surface fitting
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

print("Training Advanced XGBoost Model...")
xgb_advanced.fit(X_train_a, y_train_a)

# Evaluate
y_pred_a = xgb_advanced.predict(X_val_a)
mse_advanced = mean_squared_error(y_val_a, y_pred_a)

print(f"Original Baseline XGBoost MSE: {mse:.6f}")
print(f"Advanced Feature Engineered XGBoost MSE: {mse_advanced:.6f}")

Training Advanced XGBoost Model...
Original Baseline XGBoost MSE: 0.000580
Advanced Feature Engineered XGBoost MSE: 0.000385


In [9]:


# 1. Isolate known data and sort chronologically!
df_known_fe = df_fe[df_fe['IV'].notna()].copy()
df_known_fe = df_known_fe.sort_values(by='datetime').reset_index(drop=True)

# 2. Define the exact split index (80% point)
split_idx = int(len(df_known_fe) * 0.8)

# 3. Split chronologically (NO RANDOM SHUFFLING)
train_data = df_known_fe.iloc[:split_idx]
val_data = df_known_fe.iloc[split_idx:]

X_train_ts = train_data[features_advanced]
y_train_ts = train_data['IV']

X_val_ts = val_data[features_advanced]
y_val_ts = val_data['IV']

# 4. Train the model on the chronological split
xgb_final = xgb.XGBRegressor(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

print("Training Time-Series Validated XGBoost...")
xgb_final.fit(X_train_ts, y_train_ts)

# 5. Evaluate
y_pred_ts = xgb_final.predict(X_val_ts)
mse_ts = mean_squared_error(y_val_ts, y_pred_ts)

print(f"Realistic Time-Series MSE: {mse_ts:.6f}")

Training Time-Series Validated XGBoost...
Realistic Time-Series MSE: 0.205087


In [10]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# 1. The Clean, Sanitized Feature List (No Splines!)
features_clean = [
    'underlying_price', 'Strike', 'TTE', 'Type_Code',
    'Moneyness', 'Distance_From_ATM', 'Log_TTE', 
    'Strike_TTE_Interaction'
]

# 2. Isolate known data and sort chronologically for Time-Series Validation
df_known_clean = df_fe[df_fe['IV'].notna()].copy()
df_known_clean = df_known_clean.sort_values(by='datetime').reset_index(drop=True)

# 3. Define the exact split index (Train on first 80% of the day, Test on last 20%)
split_idx = int(len(df_known_clean) * 0.8)

# 4. Split chronologically (NO RANDOM SHUFFLING)
train_data_clean = df_known_clean.iloc[:split_idx]
val_data_clean = df_known_clean.iloc[split_idx:]

X_train_clean = train_data_clean[features_clean]
y_train_clean = train_data_clean['IV']

X_val_clean = val_data_clean[features_clean]
y_val_clean = val_data_clean['IV']

# 5. Train the model on the clean, chronological data
xgb_clean = xgb.XGBRegressor(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

print("Training Sanitized Time-Series XGBoost...")
xgb_clean.fit(X_train_clean, y_train_clean)

# 6. Evaluate
y_pred_clean = xgb_clean.predict(X_val_clean)
mse_clean = mean_squared_error(y_val_clean, y_pred_clean)

print(f"Sanitized Time-Series MSE: {mse_clean:.6f}")

Training Sanitized Time-Series XGBoost...
Sanitized Time-Series MSE: 0.231073


In [11]:
# 1. Sort by ticker and datetime to safely create time-lags
df_fe = df_fe.sort_values(by=['ticker', 'datetime']).reset_index(drop=True)

# 2. Engineer the Lag Feature (Memory)
# We take the forward-filled IV (so we don't just pull NaNs for illiquid options)
# and shift it by 1 row (looking exactly 5 minutes into the past)
df_fe['Previous_IV'] = df_fe.groupby('ticker')['IV_ffill'].shift(1)

# 3. Handle the very first row of the day (which will have no 'Previous_IV')
# We backfill just the first row so we don't lose it to NaN
df_fe['Previous_IV'] = df_fe.groupby('ticker')['Previous_IV'].fillna(method='bfill')

# 4. Update the Feature List to include Memory!
features_memory = [
    'underlying_price', 'Strike', 'TTE', 'Type_Code',
    'Moneyness', 'Distance_From_ATM', 'Log_TTE', 
    'Strike_TTE_Interaction', 'Previous_IV'
]

# THE SAME TIME-SERIES VALIDATION PIPELINE 
df_known_mem = df_fe[df_fe['IV'].notna()].copy()
df_known_mem = df_known_mem.sort_values(by='datetime').reset_index(drop=True)

split_idx = int(len(df_known_mem) * 0.8)

train_data_mem = df_known_mem.iloc[:split_idx]
val_data_mem = df_known_mem.iloc[split_idx:]

X_train_mem = train_data_mem[features_memory]
y_train_mem = train_data_mem['IV']

X_val_mem = val_data_mem[features_memory]
y_val_mem = val_data_mem['IV']

xgb_mem = xgb.XGBRegressor(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=5,
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost with Memory (Lag Features)...")
xgb_mem.fit(X_train_mem, y_train_mem)

y_pred_mem = xgb_mem.predict(X_val_mem)
mse_mem = mean_squared_error(y_val_mem, y_pred_mem)

print(f"Time-Series MSE WITH MEMORY: {mse_mem:.6f}")

Training XGBoost with Memory (Lag Features)...


/tmp/ipykernel_58/3619545923.py:11: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df_fe['Previous_IV'] = df_fe.groupby('ticker')['Previous_IV'].fillna(method='bfill')
/tmp/ipykernel_58/3619545923.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_fe['Previous_IV'] = df_fe.groupby('ticker')['Previous_IV'].fillna(method='bfill')


Time-Series MSE WITH MEMORY: 0.214142


In [12]:

import numpy as np

# 1. Memory Features (Time Lags)
# Sort chronologically by ticker to look backward in time safely
df_fe = df_fe.sort_values(by=['ticker', 'datetime']).reset_index(drop=True)
df_fe['Previous_IV'] = df_fe.groupby('ticker')['IV_ffill'].shift(1)
df_fe['Previous_IV'] = df_fe.groupby('ticker')['Previous_IV'].bfill() # Cleaned up warning

# 2. Target Variable (Delta)
df_fe['IV_Delta'] = df_fe['IV'] - df_fe['Previous_IV']

# 3. Spatial Features (Cross-Strike Lags)
# Re-sort to align the physical strikes perfectly on the grid at timestamp 't'
df_fe = df_fe.sort_values(by=['datetime', 'Type', 'Strike']).reset_index(drop=True)
df_fe['Spatial_IV_Below'] = df_fe.groupby(['datetime', 'Type'])['IV_ffill'].shift(1)
df_fe['Spatial_IV_Above'] = df_fe.groupby(['datetime', 'Type'])['IV_ffill'].shift(-1)

df_fe['Spatial_IV_Below'] = df_fe['Spatial_IV_Below'].fillna(df_fe['Previous_IV'])
df_fe['Spatial_IV_Above'] = df_fe['Spatial_IV_Above'].fillna(df_fe['Previous_IV'])
# 4. Define the Final Feature Matrix
features_memory_spatial = [
    'underlying_price', 'Strike', 'TTE', 'Type_Code',
    'Moneyness', 'Distance_From_ATM', 'Log_TTE', 
    'Strike_TTE_Interaction', 'Previous_IV',
    'Spatial_IV_Below', 'Spatial_IV_Above'
]

print("Master Feature Matrix Built Successfully.")

Master Feature Matrix Built Successfully.


In [13]:
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.interpolate import CubicSpline

print("--- INITIATING QUANTITATIVE HYBRID PIPELINE (SPLINE + ML) ---")

# 1. Fit a Cubic Spline for every single 5-minute windo
def fit_spline_per_timestamp(group):
    # Sort perfectly by strike to ensure the mathematical curve doesn't fold on itself
    group = group.sort_values('Strike')
    
    # Isolate the strikes where we know the true Implied Volatility
    known = group.dropna(subset=['IV'])
    
    # A Cubic Spline requires at least 4 known points to draw a curve safely
    if len(known) > 3:
        try:
            # bc_type='natural' forces the ends of the curve to point straight, 
            # preventing wild hallucinations in the deep Out-Of-The-Money options
            spline = CubicSpline(known['Strike'], known['IV'], bc_type='natural')
            
            # Predict the "Theoretical IV" for ALL rows in this 5-minute window
            group['Spline_IV'] = spline(group['Strike'])
        except Exception:
            # Fallback in case of bizarre market data (e.g., identical strikes)
            group['Spline_IV'] = group['Previous_IV'] 
    else:
        # If the market is too illiquid, fall back to our Spatial Lags
        group['Spline_IV'] = group['Previous_IV']
        
    return group

print("Step 1: Drawing Arbitrage-Free Mathematical Curves (Cubic Splines)...")
# Apply the function across time (This takes a few seconds)
df_hybrid = df_fe.groupby('datetime', group_keys=False).apply(fit_spline_per_timestamp)

# 2. Separate Knowns and Unknowns
df_known = df_hybrid[df_hybrid['IV'].notna()].copy()
df_unknown = df_hybrid[df_hybrid['IV'].isna()].copy()

# 3. Calculate the ML Target: The Residual (Where did the math fail?)
df_known['IV_Residual'] = df_known['IV'] - df_known['Spline_IV']

# 4. Define ML Features
X_train = df_known[features_memory_spatial]
y_train = df_known['IV_Residual']  # Notice we are predicting the Error, not the IV

X_sub = df_unknown[features_memory_spatial]

# 5. Train the ML Engine (The Sniper)
print("Step 2: Training XGBoost to find Micro-Inefficiencies...")
xgb_residual = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,      # Kept shallow because it is only finding tiny errors now
    random_state=42,
    n_jobs=-1
)
xgb_residual.fit(X_train, y_train)

# 6. Predict the Missing Residuals
print("Step 3: Executing Machine Learning Corrections...")
predicted_residuals = xgb_residual.predict(X_sub)

# 7. Reconstruct the Final Volatility Surface: Math + ML Correction
print("Step 4: Reconstructing Final Volatility Surface...")
df_unknown['Final_IV'] = df_unknown['Spline_IV'] + predicted_residuals

# Safety limits: Volatility physically cannot drop below 0.1%
df_unknown['Final_IV'] = df_unknown['Final_IV'].clip(lower=0.001)
# If Spline+ML completely failed, fallback to the Spatial Below lag
df_unknown['Final_IV'] = df_unknown['Final_IV'].fillna(df_unknown['Spatial_IV_Below']) 

# 8. Build Strict Submission Format
df_unknown['datetime_str'] = df_unknown['datetime'].dt.strftime('%d-%m-%Y %H:%M')
df_unknown['id'] = df_unknown['datetime_str'] + '||' + df_unknown['ticker']
df_unknown['value'] = df_unknown['Final_IV']

final_sub = df_unknown[['id', 'value']]

# 9. Save
final_sub.to_csv('submission.csv', index=False)
print(f"Success! Hybrid submission.csv created with shape {final_sub.shape}")
display(final_sub.head())

--- INITIATING QUANTITATIVE HYBRID PIPELINE (SPLINE + ML) ---
Step 1: Drawing Arbitrage-Free Mathematical Curves (Cubic Splines)...
Step 2: Training XGBoost to find Micro-Inefficiencies...
Step 3: Executing Machine Learning Corrections...
Step 4: Reconstructing Final Volatility Surface...
Success! Hybrid submission.csv created with shape (5460, 2)


/tmp/ipykernel_58/122701332.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_hybrid = df_fe.groupby('datetime', group_keys=False).apply(fit_spline_per_timestamp)


,id,value
17,07-01-2026 09:15||NIFTY27JAN2624100PE,0.163989
3,07-01-2026 09:15||NIFTY27JAN2625500CE,0.113581
6,07-01-2026 09:15||NIFTY27JAN2625800CE,0.100924
44,07-01-2026 09:20||NIFTY27JAN2624000PE,0.170076
46,07-01-2026 09:20||NIFTY27JAN2624200PE,0.160227
